# 🏥 Med-Bench-Arena — Colab runner for medical & TCM LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/Med-Bench-Arena/blob/main/notebooks/Med_Bench_Arena_Colab.ipynb)

Evaluate **18 medical / TCM LLMs** on bilingual medical benchmarks with a few clicks.
This notebook drives [`Med-Bench-Arena`](https://github.com/pariskang/Med-Bench-Arena): it
clones the repo, installs vLLM, lets you **pick one model**, and runs it against a small
MCQ slice (MedQA + CMB). Swap in any other config (`configs/catalog_*.yaml`) for more.

> **GPU tiers** (set `Runtime → Change runtime type`):
> - **T4 (free, 16GB)** → ≤7B models: `zhongjing-2-1_8b`, `biancang`, `biomistral-7b`,
>   `huatuogpt-o1-7b`, `clinicalgpt-r1`, `taiyi`, `lingshu-7b`, `huatuogpt2-7b`, `aquilamed-rl`.
> - **A100 (40GB)** → 13–34B: `disc-medllm`, `baichuan-m1-14b`, `dao1-30b-a3b`,
>   `deepseek-r1-32b`, `baichuan-m2-32b`, `medgemma-27b-it`.
> - **2×A100 / H100 80GB** → 70B: `meditron-70b`, `citrus-70b`, `huatuogpt-o1-70b/72b`.
>
> One model per session — vLLM keeps the model resident in GPU memory.

## 1 · Check the GPU

In [ ]:
!nvidia-smi

## 2 · Install Med-Bench-Arena + vLLM

Takes a few minutes (vLLM + torch). Restart the runtime only if Colab asks.

In [ ]:
# Clone the repo (use your fork/branch if needed)
![ -d Med-Bench-Arena ] || git clone https://github.com/pariskang/Med-Bench-Arena.git
%cd Med-Bench-Arena

# Core package (no GPU) + the HF/vLLM backend.
!pip install -q -e .
!pip install -q 'datasets>=2.18'
# vLLM pulls a compatible torch/transformers. peft is needed for the ZhongJing LoRA.
!pip install -q vllm peft
print('\n✅ install done — if Colab shows a "restart" banner, restart then re-run from here.')

## 3 · (Optional) HuggingFace login — only for GATED models

`meditron-70b` and `medgemma-27b-it` are gated: accept their license on the HF model
page first, then paste a token (https://huggingface.co/settings/tokens). Skip for the rest.

In [ ]:
from huggingface_hub import notebook_login
# notebook_login()   # uncomment for gated models (Meditron / MedGemma)

## 4 · Pick a model

`MODEL_ID` must be one of the ids in `configs/catalog_med_models.yaml` (printed below).
Every repo id / dtype / context / trust_remote_code there was verified against the live
HF page — see [`MODELS.md`](https://github.com/pariskang/Med-Bench-Arena/blob/main/MODELS.md).

In [ ]:
import yaml
CONFIG = 'configs/catalog_med_models.yaml'
cfg = yaml.safe_load(open(CONFIG))
ids = [m['id'] for m in cfg['models'] if m.get('type') == 'hf']
print('Available HF model ids:')
for i in ids: print('  -', i)

In [ ]:
#@title Selection
MODEL_ID = 'zhongjing-2-1_8b'  #@param {type:'string'}
LIMIT    = 20                  #@param {type:'integer'}   # samples per dataset (None = all)
GPU_MEM_UTIL = 0.90            #@param {type:'number'}    # lower to 0.80 if you OOM on load

## 5 · Run the evaluation

Filters the catalog to your one model, applies `--limit`, and runs `medeval`. Generations
are cached under `results/med_models/cache/`, so re-runs are cheap.

In [ ]:
import copy, medeval

spec = next(m for m in cfg['models'] if m['id'] == MODEL_ID)
run_cfg = copy.deepcopy(cfg)
# keep only the picked model + any judge_only models (datasets may need a judge)
run_cfg['models'] = [m for m in cfg['models']
                     if m['id'] == MODEL_ID or m.get('judge_only')]
for m in run_cfg['models']:
    if m['id'] == MODEL_ID:
        m['gpu_memory_utilization'] = GPU_MEM_UTIL
if LIMIT:
    for d in run_cfg['datasets']:
        d['limit'] = LIMIT

print(f'Running {MODEL_ID}  ({spec["model"]})  on', [d['id'] for d in run_cfg['datasets']])
rows = medeval.run_config(run_cfg)

## 6 · Results

In [ ]:
from pathlib import Path
print(Path('results/med_models/leaderboard.md').read_text())

In [ ]:
# Peek at a few graded samples (prediction vs gold)
import json, glob
detail = sorted(glob.glob('results/med_models/detail__*.jsonl'))
for f in detail:
    print('\n===', f, '===')
    for line in open(f).readlines()[:2]:
        r = json.loads(line)
        print('Q   :', r['prompt'][:160].replace('\n',' '))
        print('pred:', str(r['parsed'])[:80], '| gold:', r['reference'])
        print('score:', {k:v['value'] for k,v in r['scores'].items()})

## 7 · Notes & next steps

- **More benchmarks**: point `CONFIG` at `configs/catalog_mcq.yaml` (English+CN+TCM MCQ),
  `configs/catalog_cn_tcm.yaml`, `configs/catalog_en_med.yaml`, `configs/example_tcm.yaml`
  (辨证/方剂/安全, needs a judge), or `configs/catalog_ethics_safety.yaml`.
- **Open-ended / safety scoring** needs a real judge. Add to `run_cfg['models']`:
  `{'id':'deepseek-r1','type':'litellm','model':'deepseek/deepseek-reasoner',`
  `'api_key_env':'DEEPSEEK_API_KEY','judge_only':True}` and set `os.environ['DEEPSEEK_API_KEY']`.
- **Big models on small GPUs**: 30–70B won't fit a single T4/A100-40GB at bf16. Use a bigger
  runtime, set `tensor_parallel` to the GPU count, or serve a 4-bit/AWQ/GPTQ community quant
  (point the model id at the quantized repo).
- **Multimodal** (`lingshu-7b`, `medgemma-27b-it`, `qilin-med-vl`): they run here as text
  backends for MCQ. For image benchmarks (舌象/影像) use `configs/catalog_multimodal.yaml`.
- **Reasoning models** (`deepseek-r1-32b`, `huatuogpt-o1-*`, `clinicalgpt-r1`, `baichuan-m2-32b`)
  emit a long think trace; `max_tokens` is set to 2048 so the final answer isn't truncated.